In [ ]:
import os
import sys

# Mount Google Drive if running in Colab
try:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
    REPO_ROOT = '/content/drive/MyDrive/Stocks'
except ImportError:
    REPO_ROOT = os.path.dirname(os.path.abspath('__file__'))

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import yfinance as yf
from IPython.display import HTML, display

from portfolio.allocations import (
    TARGET_WEIGHTS, MONTHLY_DEPOSIT,
    NUCLEAR_BASKET_TARGETS, QUANTUM_BASKET_TARGETS, CYBER_BASKET_TARGETS,
    my_current_shares,
)
from portfolio.crypto import (
    CRYPTO_TARGET_WEIGHTS, my_current_crypto, MONTHLY_DEPOSIT_CHF,
)
from portfolio.helpers import get_price, get_live_fx_rate

# =========================================================================
# 1. FETCH LIVE PRICES
# =========================================================================
print('Fetching stock/ETF prices...')
stock_prices = {t: get_price(t) for t in my_current_shares.keys()}

print('Fetching crypto prices...')
crypto_prices_usd = {}
for ticker in CRYPTO_TARGET_WEIGHTS:
    try:
        data = yf.Ticker(ticker).history(period='1d')
        crypto_prices_usd[ticker] = float(data['Close'].iloc[-1]) if not data.empty else 0.01
    except Exception:
        crypto_prices_usd[ticker] = 0.01

print('Fetching FX rates...')
USD_TO_CHF = get_live_fx_rate('USD', 'CHF')
EUR_TO_CHF = get_live_fx_rate('EUR', 'CHF')

# =========================================================================
# 2. COMPUTE PORTFOLIO VALUES
# =========================================================================

# --- Stocks / ETFs ---
stock_values = {t: shares * stock_prices.get(t, 0) for t, shares in my_current_shares.items()}

nuclear_val = sum(stock_values.get(t, 0) for t in NUCLEAR_BASKET_TARGETS)
quantum_val = sum(stock_values.get(t, 0) for t in QUANTUM_BASKET_TARGETS)
cyber_val = sum(stock_values.get(t, 0) for t in CYBER_BASKET_TARGETS)

macro_values = {
    'VWCE.DE': stock_values.get('VWCE.DE', 0),
    'XAIX.DE': stock_values.get('XAIX.DE', 0),
    'SMH': stock_values.get('SMH', 0),
    'IUIT.L': stock_values.get('IUIT.L', 0),
    'NUCLEAR_SATELLITE': nuclear_val,
    'QUANTUM_SATELLITE': quantum_val,
    'CYBER_SATELLITE': cyber_val,
}
total_stock_val = sum(macro_values.values())

# --- Crypto ---
crypto_values_usd = {
    c: my_current_crypto.get(c, 0) * crypto_prices_usd.get(c, 0)
    for c in CRYPTO_TARGET_WEIGHTS
}
total_crypto_usd = sum(crypto_values_usd.values())
total_crypto_chf = total_crypto_usd * USD_TO_CHF

# --- Grand total (approximate, mixed currencies) ---
total_portfolio_usd = total_stock_val + total_crypto_usd

# =========================================================================
# 3. BUILD HTML REPORT
# =========================================================================

def fmt_money(val):
    if abs(val) >= 1e6:
        return f'${val/1e6:,.2f}M'
    elif abs(val) >= 1e3:
        return f'${val:,.0f}'
    return f'${val:,.2f}'

def pct(val, total):
    return f'{(val/total*100):.1f}%' if total > 0 else '0.0%'

def drift_cell(current_pct, target_pct):
    diff = current_pct - target_pct
    if abs(diff) < 2:
        color = '#1B5E20'  # green — on target
    elif abs(diff) < 5:
        color = '#E65100'  # orange — drifting
    else:
        color = '#B71C1C'  # red — needs rebalance
    return f'<td style="color:{color}; font-weight:bold;">{diff:+.1f}%</td>'

# --- Macro labels ---
MACRO_LABELS = {
    'VWCE.DE': 'VWCE.DE — Global Safety Anchor',
    'XAIX.DE': 'XAIX.DE — AI & Big Data Index',
    'SMH': 'SMH — Semiconductors',
    'IUIT.L': 'IUIT.L — S&P 500 Info Tech',
    'NUCLEAR_SATELLITE': 'Nuclear Satellite (CCJ, GEV, SRUUF, LEU, SMR, OKLO)',
    'QUANTUM_SATELLITE': 'Quantum Satellite (IONQ, QNT, QBTS, RGTI, QUBT)',
    'CYBER_SATELLITE': 'Cyber Satellite (CRWD, PANW)',
}

MACRO_COLORS = {
    'VWCE.DE': '#E8F5E9', 'XAIX.DE': '#E3F2FD', 'SMH': '#E3F2FD',
    'IUIT.L': '#E3F2FD', 'NUCLEAR_SATELLITE': '#FFF8DC',
    'QUANTUM_SATELLITE': '#F3E6F5', 'CYBER_SATELLITE': '#FFEBEE',
}

CRYPTO_COLORS = {
    'BTC-USD': '#FFF3E0', 'ETH-USD': '#E3F2FD', 'SOL-USD': '#F3E6F5',
    'RENDER-USD': '#E8F5E9', 'LINK-USD': '#E3F2FD', 'XRP-USD': '#ECEFF1',
}

CRYPTO_LABELS = {
    'BTC-USD': 'Bitcoin', 'ETH-USD': 'Ethereum', 'SOL-USD': 'Solana',
    'RENDER-USD': 'Render', 'LINK-USD': 'Chainlink', 'XRP-USD': 'XRP',
}

html = []
html.append('<style>')
html.append('  .po-table { border-collapse: collapse; width: 100%; font-family: Arial, sans-serif; font-size: 13px; }')
html.append('  .po-table th { background: #2F4F4F; color: white; padding: 8px 12px; text-align: left; }')
html.append('  .po-table td { padding: 6px 12px; border-bottom: 1px solid #ddd; }')
html.append('  .po-table tr:hover { background: #f5f5f5 !important; }')
html.append('  .po-section { background: white; padding: 20px; border-radius: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.1); margin-bottom: 20px; overflow-x: auto; }')
html.append('  .po-header { font-size: 18px; font-weight: bold; margin-bottom: 12px; }')
html.append('  .po-summary { display: flex; gap: 20px; margin-bottom: 20px; flex-wrap: wrap; }')
html.append('  .po-card { background: white; padding: 16px 24px; border-radius: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.1); text-align: center; min-width: 180px; }')
html.append('  .po-card-value { font-size: 24px; font-weight: bold; color: #2F4F4F; }')
html.append('  .po-card-label { font-size: 12px; color: #666; margin-top: 4px; }')
html.append('</style>')

# --- Summary cards ---
stock_pct = total_stock_val / total_portfolio_usd * 100 if total_portfolio_usd > 0 else 0
crypto_pct = total_crypto_usd / total_portfolio_usd * 100 if total_portfolio_usd > 0 else 0

html.append('<div class="po-summary">')
html.append(f'<div class="po-card"><div class="po-card-value">{fmt_money(total_portfolio_usd)}</div><div class="po-card-label">Total Portfolio (approx USD)</div></div>')
html.append(f'<div class="po-card"><div class="po-card-value">{fmt_money(total_stock_val)}</div><div class="po-card-label">Stocks & ETFs ({stock_pct:.0f}%)</div></div>')
html.append(f'<div class="po-card"><div class="po-card-value">{fmt_money(total_crypto_usd)}</div><div class="po-card-label">Crypto ({crypto_pct:.0f}%)</div></div>')
html.append(f'<div class="po-card"><div class="po-card-value">{USD_TO_CHF:.4f}</div><div class="po-card-label">USD/CHF</div></div>')
html.append(f'<div class="po-card"><div class="po-card-value">{EUR_TO_CHF:.4f}</div><div class="po-card-label">EUR/CHF</div></div>')
html.append('</div>')

# =========================================================================
# STOCKS & ETFs TABLE
# =========================================================================
html.append('<div class="po-section">')
html.append('<div class="po-header">Stocks & ETFs Allocation</div>')
html.append('<table class="po-table">')
html.append('<tr><th>Asset</th><th>Value</th><th>Current %</th><th>Target %</th><th>Drift</th></tr>')

for asset, target_pct in TARGET_WEIGHTS.items():
    val = macro_values.get(asset, 0)
    cur_pct = val / total_stock_val * 100 if total_stock_val > 0 else 0
    tgt_pct = target_pct * 100
    label = MACRO_LABELS.get(asset, asset)
    bg = MACRO_COLORS.get(asset, '#FFFFFF')
    html.append(f'<tr style="background:{bg};">')
    html.append(f'<td><b>{label}</b></td>')
    html.append(f'<td>{fmt_money(val)}</td>')
    html.append(f'<td>{cur_pct:.1f}%</td>')
    html.append(f'<td>{tgt_pct:.0f}%</td>')
    html.append(drift_cell(cur_pct, tgt_pct))
    html.append('</tr>')

# Total row
html.append(f'<tr style="background:#2F4F4F; color:white; font-weight:bold;">')
html.append(f'<td>TOTAL</td><td>{fmt_money(total_stock_val)}</td><td>100%</td><td>100%</td><td>—</td></tr>')
html.append('</table>')

# --- Sub-basket detail ---
html.append('<br><table class="po-table">')
html.append('<tr><th>Ticker</th><th>Basket</th><th>Shares</th><th>Price</th><th>Value</th><th>Basket %</th><th>Target %</th><th>Drift</th></tr>')

baskets = [
    ('Nuclear', NUCLEAR_BASKET_TARGETS, nuclear_val),
    ('Quantum', QUANTUM_BASKET_TARGETS, quantum_val),
    ('Cyber', CYBER_BASKET_TARGETS, cyber_val),
]
basket_colors = {'Nuclear': '#FFF8DC', 'Quantum': '#F3E6F5', 'Cyber': '#FFEBEE'}

for basket_name, targets, basket_total in baskets:
    for ticker, target in targets.items():
        shares = my_current_shares.get(ticker, 0)
        price = stock_prices.get(ticker, 0)
        val = stock_values.get(ticker, 0)
        cur_pct = val / basket_total * 100 if basket_total > 0 else 0
        tgt_pct = target * 100
        bg = basket_colors.get(basket_name, '#FFFFFF')
        html.append(f'<tr style="background:{bg};">')
        html.append(f'<td><b>{ticker}</b></td>')
        html.append(f'<td>{basket_name}</td>')
        html.append(f'<td>{shares}</td>')
        html.append(f'<td>${price:,.2f}</td>')
        html.append(f'<td>{fmt_money(val)}</td>')
        html.append(f'<td>{cur_pct:.1f}%</td>')
        html.append(f'<td>{tgt_pct:.0f}%</td>')
        html.append(drift_cell(cur_pct, tgt_pct))
        html.append('</tr>')

html.append('</table>')
html.append('</div>')

# =========================================================================
# CRYPTO TABLE
# =========================================================================
html.append('<div class="po-section">')
html.append(f'<div class="po-header">Crypto Allocation (CHF Denominated — 1 USD = {USD_TO_CHF:.4f} CHF)</div>')
html.append('<table class="po-table">')
html.append('<tr><th>Asset</th><th>Holdings</th><th>Price (USD)</th><th>Value (USD)</th><th>Value (CHF)</th><th>Current %</th><th>Target %</th><th>Drift</th></tr>')

for ticker, target_pct in CRYPTO_TARGET_WEIGHTS.items():
    holdings = my_current_crypto.get(ticker, 0)
    price = crypto_prices_usd.get(ticker, 0)
    val_usd = crypto_values_usd.get(ticker, 0)
    val_chf = val_usd * USD_TO_CHF
    cur_pct = val_usd / total_crypto_usd * 100 if total_crypto_usd > 0 else 0
    tgt_pct = target_pct * 100
    label = CRYPTO_LABELS.get(ticker, ticker)
    bg = CRYPTO_COLORS.get(ticker, '#FFFFFF')
    html.append(f'<tr style="background:{bg};">')
    html.append(f'<td><b>{ticker}</b> ({label})</td>')
    html.append(f'<td>{holdings:,.4f}</td>')
    html.append(f'<td>${price:,.2f}</td>')
    html.append(f'<td>{fmt_money(val_usd)}</td>')
    html.append(f'<td>{val_chf:,.0f} CHF</td>')
    html.append(f'<td>{cur_pct:.1f}%</td>')
    html.append(f'<td>{tgt_pct:.0f}%</td>')
    html.append(drift_cell(cur_pct, tgt_pct))
    html.append('</tr>')

# Total row
html.append(f'<tr style="background:#2F4F4F; color:white; font-weight:bold;">')
html.append(f'<td>TOTAL</td><td></td><td></td><td>{fmt_money(total_crypto_usd)}</td><td>{total_crypto_chf:,.0f} CHF</td><td>100%</td><td>100%</td><td>—</td></tr>')
html.append('</table>')

# --- Buy recommendation ---
if total_crypto_usd > 0:
    chosen = max(CRYPTO_TARGET_WEIGHTS, key=lambda c: CRYPTO_TARGET_WEIGHTS[c] - (crypto_values_usd.get(c, 0) / total_crypto_usd))
    price_chf = crypto_prices_usd[chosen] * USD_TO_CHF
    units = MONTHLY_DEPOSIT_CHF / price_chf if price_chf > 0 else 0
    html.append(f'<p style="margin-top:12px; font-size:14px;">'
                f'<b>Monthly Buy Recommendation:</b> {chosen} ({CRYPTO_LABELS.get(chosen, "")}) — '
                f'{units:.4f} tokens at {price_chf:,.2f} CHF = <b>{MONTHLY_DEPOSIT_CHF:,.0f} CHF</b></p>')

# --- Stock buy recommendation ---
if total_stock_val > 0:
    chosen_stock = max(TARGET_WEIGHTS, key=lambda a: TARGET_WEIGHTS[a] - (macro_values.get(a, 0) / total_stock_val))
    html.append(f'<p style="font-size:14px;">'
                f'<b>Monthly Stock/ETF Buy:</b> {MACRO_LABELS.get(chosen_stock, chosen_stock)} — '
                f'<b>€{MONTHLY_DEPOSIT:,.0f}</b></p>')

html.append('</div>')

# =========================================================================
# 4. RENDER
# =========================================================================
display(HTML('\n'.join(html)))
print('\nPortfolio overview rendered.')